In [1]:
import pandas as pd

In [2]:
df = pd.read_parquet("yellow_tripdata_2026-01.parquet")

In [3]:
df.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,2,2026-01-01 00:54:04,2026-01-01 00:59:37,1.0,0.97,1.0,N,239,238,1,7.2,1.00,0.5,3.66,0.0,1.0,15.86,2.5,0.0,0.00
1,1,2026-01-01 00:34:04,2026-01-01 00:39:47,0.0,0.90,1.0,N,163,162,2,7.9,4.25,0.5,0.00,0.0,1.0,13.65,2.5,0.0,0.75
2,1,2026-01-01 00:57:06,2026-01-01 01:05:59,0.0,1.40,1.0,N,43,237,1,10.7,4.25,0.5,2.50,0.0,1.0,18.95,2.5,0.0,0.75
3,2,2026-01-01 00:15:22,2026-01-01 00:58:10,4.0,5.58,1.0,N,142,209,1,38.7,1.00,0.5,11.11,0.0,1.0,55.56,2.5,0.0,0.75
4,2,2026-01-01 00:27:13,2026-01-01 00:40:43,0.0,2.16,1.0,N,88,144,1,13.5,1.00,0.5,3.85,0.0,1.0,23.10,2.5,0.0,0.75


In [4]:
df.shape

(3724889, 20)

In [5]:
df.isnull().sum()

VendorID                       0
tpep_pickup_datetime           0
tpep_dropoff_datetime          0
passenger_count          1088058
trip_distance                  0
RatecodeID               1088058
store_and_fwd_flag       1088058
PULocationID                   0
DOLocationID                   0
payment_type                   0
fare_amount                    0
extra                          0
mta_tax                        0
tip_amount                     0
tolls_amount                   0
improvement_surcharge          0
total_amount                   0
congestion_surcharge     1088058
Airport_fee              1088058
cbd_congestion_fee             0
dtype: int64

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3724889 entries, 0 to 3724888
Data columns (total 20 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   VendorID               int32         
 1   tpep_pickup_datetime   datetime64[us]
 2   tpep_dropoff_datetime  datetime64[us]
 3   passenger_count        float64       
 4   trip_distance          float64       
 5   RatecodeID             float64       
 6   store_and_fwd_flag     object        
 7   PULocationID           int32         
 8   DOLocationID           int32         
 9   payment_type           int64         
 10  fare_amount            float64       
 11  extra                  float64       
 12  mta_tax                float64       
 13  tip_amount             float64       
 14  tolls_amount           float64       
 15  improvement_surcharge  float64       
 16  total_amount           float64       
 17  congestion_surcharge   float64       
 18  Airport_fee           

In [7]:
missing_cols = [
    "passenger_count",
    "RatecodeID",
    "store_and_fwd_flag",
    "congestion_surcharge",
    "Airport_fee"
]

# Nombre de lignes où les 5 colonnes sont manquantes
print(df[missing_cols].isna().all(axis=1).sum())

# Nombre de lignes où au moins une colonne est manquante
print(df[missing_cols].isna().any(axis=1).sum())

1088058
1088058


In [8]:
# Vérifier si les 5 colonnes sont toujours manquantes ensemble
df[missing_cols].isna().sum(axis=1).value_counts().sort_index()

0    2636831
5    1088058
Name: count, dtype: int64

In [9]:
print("Nombre de doublons :", df.duplicated().sum())

Nombre de doublons : 0


In [10]:
df["trip_duration"] = (
    df["tpep_dropoff_datetime"]
    - df["tpep_pickup_datetime"]
).dt.total_seconds() / 60

df["trip_duration"].describe()

count    3.724889e+06
mean     1.719316e+01
std      2.517802e+01
min     -1.170000e+01
25%      8.066667e+00
50%      1.336667e+01
75%      2.120000e+01
max      7.507900e+03
Name: trip_duration, dtype: float64

In [11]:
print("Durées négatives :", (df["trip_duration"] < 0).sum())

print("Durées nulles :", (df["trip_duration"] == 0).sum())

Durées négatives : 1
Durées nulles : 45069


In [15]:
df[df["trip_duration"] < 0][
    ["tpep_pickup_datetime",
     "tpep_dropoff_datetime",
     "trip_distance",
     "fare_amount",
     "total_amount"]
]

,tpep_pickup_datetime,tpep_dropoff_datetime,trip_distance,fare_amount,total_amount
1465434,2026-01-17 19:30:00,2026-01-17 19:18:18,16.5,66.5,67.0


In [13]:
df[df["trip_duration"] == 0][
    ["tpep_pickup_datetime",
     "tpep_dropoff_datetime",
     "trip_distance",
     "fare_amount",
     "total_amount"]
].head(10)

,tpep_pickup_datetime,tpep_dropoff_datetime,trip_distance,fare_amount,total_amount
16,2026-01-01 00:45:15,2026-01-01 00:45:15,1.54,28.9,41.58
126,2026-01-01 00:46:05,2026-01-01 00:46:05,1.79,18.4,24.15
251,2026-01-01 00:22:40,2026-01-01 00:22:40,1.24,8.6,16.32
252,2026-01-01 00:36:04,2026-01-01 00:36:04,2.00,11.4,17.15
253,2026-01-01 00:51:23,2026-01-01 00:51:23,0.83,6.5,12.25
287,2026-01-01 00:48:27,2026-01-01 00:48:27,3.83,27.5,39.90
382,2026-01-01 00:45:00,2026-01-01 00:45:00,5.38,27.5,32.50
405,2026-01-01 00:51:33,2026-01-01 00:51:33,1.22,10.0,18.90
511,2026-01-01 00:40:03,2026-01-01 00:40:03,1.69,19.8,30.66
527,2026-01-01 00:36:07,2026-01-01 00:36:07,1.13,7.9,16.38


In [16]:
zero_duration = df[df["trip_duration"] == 0]

print(
    "Distance positive :",
    (zero_duration["trip_distance"] > 0).sum()
)

print(
    "Distance nulle :",
    (zero_duration["trip_distance"] == 0).sum()
)

print(
    "Distance négative :",
    (zero_duration["trip_distance"] < 0).sum()
)

Distance positive : 43905
Distance nulle : 1164
Distance négative : 0


In [17]:
#verification de la qualite des donnees sur la distance 
print(
    "Distances négatives :",
    (df["trip_distance"] < 0).sum()
)

print(
    "Distances nulles :",
    (df["trip_distance"] == 0).sum()
)

Distances négatives : 0
Distances nulles : 125738


In [19]:
#savoir si les trajets de distance nulle ont quand même généré des frais.
zero_distance = df[df["trip_distance"] == 0]

print("Nombre de trajets :", len(zero_distance))

print(
    "Montant total positif :",
    (zero_distance["total_amount"] > 0).sum()
)

print(
    "Montant total nul :",
    (zero_distance["total_amount"] == 0).sum()
)

print(
    "Montant total négatif :",
    (zero_distance["total_amount"] < 0).sum()
)

Nombre de trajets : 125738
Montant total positif : 122657
Montant total nul : 172
Montant total négatif : 2909


In [22]:
zero_distance[
    [
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "trip_duration",
        "fare_amount",
        "trip_distance",
        "total_amount",
        "payment_type"
    ]
].head(10)

,tpep_pickup_datetime,tpep_dropoff_datetime,trip_duration,fare_amount,trip_distance,total_amount,payment_type
118,2026-01-01 00:21:36,2026-01-01 00:22:52,1.266667,3.0,0.0,8.00,2
329,2026-01-01 00:11:22,2026-01-01 00:34:04,22.700000,18.4,0.0,24.15,1
330,2026-01-01 00:54:54,2026-01-01 01:09:08,14.233333,12.8,0.0,23.15,1
360,2026-01-01 00:32:10,2026-01-01 00:32:14,0.066667,50.0,0.0,54.25,1
634,2026-01-01 00:45:48,2026-01-01 00:45:54,0.100000,50.0,0.0,54.25,1
681,2026-01-01 00:56:10,2026-01-01 00:56:24,0.233333,3.0,0.0,8.75,2
902,2026-01-01 00:35:40,2026-01-01 00:39:04,3.400000,5.1,0.0,7.60,2
1159,2026-01-01 00:11:44,2026-01-01 00:11:48,0.066667,40.0,0.0,41.00,1
1270,2026-01-01 00:40:52,2026-01-01 00:40:58,0.100000,124.0,0.0,130.00,1
1289,2026-01-01 00:28:16,2026-01-01 00:28:21,0.083333,60.0,0.0,62.08,1


In [23]:
#On va regarder si les courses à distance nulle proviennent davantage d'un fournisseur, d'une zone ou d'un type de tarif.
zero_distance = df[df["trip_distance"] == 0]

print("Répartition par fournisseur :")
print(zero_distance["VendorID"].value_counts())

print("\nRépartition par type de paiement :")
print(zero_distance["payment_type"].value_counts())

print("\nRépartition par code tarifaire :")
print(zero_distance["RatecodeID"].value_counts(dropna=False))

print("\nRépartition par zone de départ :")
print(zero_distance["PULocationID"].value_counts().head(10))

Répartition par fournisseur :
VendorID
2    114960
1      9953
7       825
Name: count, dtype: int64

Répartition par type de paiement :
payment_type
0    92717
1    18249
2     7643
4     3889
3     3240
Name: count, dtype: int64

Répartition par code tarifaire :
RatecodeID
NaN     92717
1.0     17940
5.0     11415
2.0      2418
3.0       647
99.0      583
4.0        15
6.0         3
Name: count, dtype: int64

Répartition par zone de départ :
PULocationID
236    3527
161    3392
246    3304
234    3192
237    3130
79     3113
170    3046
68     2909
132    2823
140    2694
Name: count, dtype: int64


In [29]:
# Taux de distance nulle par fournisseur
vendor_analysis = df.groupby("VendorID").agg(
    total_trips=("trip_distance", "size"),
    zero_distance=("trip_distance", lambda x: (x == 0).sum())
)

vendor_analysis["zero_distance_rate"] = (
    vendor_analysis["zero_distance"] / vendor_analysis["total_trips"] * 100
)

print(vendor_analysis.sort_values("zero_distance_rate", ascending=False))

          total_trips  zero_distance  zero_distance_rate
VendorID                                                
2             2965742         114960            3.876264
7               44705            825            1.845431
1              710425           9953            1.400992
6                4017              0            0.000000


In [30]:
zero_distance = df[df["trip_distance"] == 0]

print("Distance nulle ET RatecodeID manquant :",
      zero_distance["RatecodeID"].isna().sum())

print("Distance nulle ET payment_type = 0 :",
      (zero_distance["payment_type"] == 0).sum())

print("Distance nulle ET les deux conditions :",
      (
          zero_distance["RatecodeID"].isna()
          & (zero_distance["payment_type"] == 0)
      ).sum())

Distance nulle ET RatecodeID manquant : 92717
Distance nulle ET payment_type = 0 : 92717
Distance nulle ET les deux conditions : 92717


In [31]:
#Quel est le pourcentage de trajets ayant une distance nulle selon le type de paiement ?
pd.crosstab(
    df["payment_type"],
    df["trip_distance"].eq(0),
    normalize="index"
) * 100

trip_distance,False,True
payment_type,,
0,91.478671,8.521329
1,99.188842,0.811158
2,97.566257,2.433743
3,80.530016,19.469984
4,93.104610,6.895390


In [32]:
free_zero_distance = df[
    (df["payment_type"] == 3) &
    (df["trip_distance"] == 0)
]

print("Nombre de courses :", len(free_zero_distance))

print(
    free_zero_distance[
        [
            "tpep_pickup_datetime",
            "tpep_dropoff_datetime",
            "trip_duration",
            "fare_amount",
            "total_amount",
            "PULocationID",
            "DOLocationID"
        ]
    ].head(10)
)

print("\nRépartition des montants :")
print(free_zero_distance[["fare_amount", "total_amount"]].describe())

Nombre de courses : 3240
     tpep_pickup_datetime tpep_dropoff_datetime  trip_duration  fare_amount  \
2037  2026-01-01 00:09:53   2026-01-01 00:09:59       0.100000          3.0   
2622  2026-01-01 00:22:12   2026-01-01 00:23:08       0.933333          3.0   
4654  2026-01-01 00:32:09   2026-01-01 00:32:21       0.200000         -3.0   
4655  2026-01-01 00:32:09   2026-01-01 00:32:21       0.200000          3.0   
4724  2026-01-01 00:40:46   2026-01-01 00:40:52       0.100000         -3.0   
4725  2026-01-01 00:40:46   2026-01-01 00:40:52       0.100000          3.0   
5003  2026-01-01 00:05:53   2026-01-01 00:06:51       0.966667          3.0   
6317  2026-01-01 01:06:27   2026-01-01 01:08:06       1.650000          3.7   
7726  2026-01-01 01:22:45   2026-01-01 01:24:31       1.766667          3.7   
7854  2026-01-01 01:00:27   2026-01-01 01:00:47       0.333333         -3.0   

      total_amount  PULocationID  DOLocationID  
2037          5.50            25            25  
2622   

In [33]:
# Répartition des montants totaux
print(
    free_zero_distance["total_amount"]
    .apply(
        lambda x: "Négatif" if x < 0
        else "Nul" if x == 0
        else "Positif"
    )
    .value_counts()
)

total_amount
Positif    2562
Négatif     609
Nul          69
Name: count, dtype: int64


In [34]:
# Nombre de lignes appartenant à des groupes
# ayant les mêmes horaires et les mêmes zones
keys = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID"
]

duplicate_groups = free_zero_distance[
    free_zero_distance.duplicated(subset=keys, keep=False)
]

print("Lignes appartenant à des groupes similaires :",
      len(duplicate_groups))

print(
    duplicate_groups[
        keys + ["fare_amount", "total_amount"]
    ].sort_values(keys).head(20)
)

Lignes appartenant à des groupes similaires : 1122
      tpep_pickup_datetime tpep_dropoff_datetime  PULocationID  DOLocationID  \
4654   2026-01-01 00:32:09   2026-01-01 00:32:21           132           132   
4655   2026-01-01 00:32:09   2026-01-01 00:32:21           132           132   
4724   2026-01-01 00:40:46   2026-01-01 00:40:52           144           144   
4725   2026-01-01 00:40:46   2026-01-01 00:40:52           144           144   
7854   2026-01-01 01:00:27   2026-01-01 01:00:47            90            90   
7855   2026-01-01 01:00:27   2026-01-01 01:00:47            90            90   
12246  2026-01-01 02:14:00   2026-01-01 02:14:12           170           170   
12247  2026-01-01 02:14:00   2026-01-01 02:14:12           170           170   
14409  2026-01-01 03:00:35   2026-01-01 03:00:46            48            48   
14410  2026-01-01 03:00:35   2026-01-01 03:00:46            48            48   
19367  2026-01-01 05:27:10   2026-01-01 05:27:26            79       

In [37]:
#vérifier si les montants négatifs et positifs s'annulent exactement au sein de ces groupes.
group_analysis = (
    free_zero_distance
    .groupby(keys)
    .agg(
        number_of_records=("total_amount", "size"),
        total_fare=("fare_amount", "sum"),
        total_amount=("total_amount", "sum")
    )
    .reset_index()
)

# Groupes contenant plusieurs enregistrements
group_analysis = group_analysis[
    group_analysis["number_of_records"] > 1
]

print("Nombre de groupes :", len(group_analysis))

print(
    "Groupes dont le total_amount s'annule :",
    (group_analysis["total_amount"].abs() < 0.001).sum()
)

print(
    "Groupes dont le fare_amount s'annule :",
    (group_analysis["total_fare"].abs() < 0.001).sum()
)

print(group_analysis.head(10))

Nombre de groupes : 561
Groupes dont le total_amount s'annule : 561
Groupes dont le fare_amount s'annule : 561
   tpep_pickup_datetime tpep_dropoff_datetime  PULocationID  DOLocationID  \
3   2026-01-01 00:32:09   2026-01-01 00:32:21           132           132   
4   2026-01-01 00:40:46   2026-01-01 00:40:52           144           144   
5   2026-01-01 01:00:27   2026-01-01 01:00:47            90            90   
14  2026-01-01 02:14:00   2026-01-01 02:14:12           170           170   
18  2026-01-01 03:00:35   2026-01-01 03:00:46            48            48   
26  2026-01-01 05:27:10   2026-01-01 05:27:26            79            79   
31  2026-01-01 06:19:59   2026-01-01 06:20:16           162           162   
34  2026-01-01 08:27:26   2026-01-01 08:28:41           158           158   
46  2026-01-01 12:24:03   2026-01-01 12:24:09           162           162   
51  2026-01-01 15:29:41   2026-01-01 15:29:47           138           138   

    number_of_records  total_fare  total_